In [ ]:
# Установка необходимых библиотек
!pip install requests huggingface-hub transformers accelerate --quiet

import json
import requests
import getpass
from datetime import datetime
from typing import Dict, List
import torch
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import pipeline

# ===================== КОНФИГУРАЦИЯ =====================
# Ваш Hugging Face токен
HF_TOKEN = "token"

# Авторизация
try:
    login(token=HF_TOKEN)
    print("✅ Успешная авторизация в Hugging Face!")
except Exception as e:
    print(f"⚠️ Ошибка авторизации: {e}")

PRODUCT_DESCRIPTION = "Создание сайтов-визиток для малого бизнеса. Целевая аудитория: владельцы небольших салонов красоты, кофеен, пекарен, у которых нет сайта или он устаревший."

PROMPT_TEMPLATE = f"""
Ты — AI-ассистент по продажам. Твоя задача — проводить холодные outreach-рассылки в Telegram для привлечения клиентов из малого бизнеса.
**Твои ключевые правила:**
1.  **Цель:** Не продавать сразу, а заинтересовать и получить согласие на передачу контакта менеджеру ("теплый" лид).
2.  **Тон:** Дружелюбный, краткий, деловой, но неформальный. Как общение в мессенджере.
3.  **Структура сообщения:**
    *   **Приветствие и представление.**
    *   **Короткое указание на потенциальную проблему/боль** целевой аудитории.
    *   **Предложение ценности** — как твой продукт/услуга решает эту боль.
    *   **Призыв к действию (CTA):** Предложить перейти на сайт, записаться на короткий звонок или оставить контакты.
4.  **Важно:** Не будь спамным, не навязывайся. Пиши так, как писал бы человек.
5.  **Контекст:** Ты пишешь первым, это холодное сообщение в Telegram.

**Продукт/услуга:** {PRODUCT_DESCRIPTION}

Напиши холодное сообщение для Telegram для {{target_audience}}.
Формат ответа: Только текст сообщения, которое будет отправлено клиенту. Никаких лишних пояснений.
"""

TEST_CASES = [
    "Владелец салона красоты",
    "Владелец кофейни",
    "Владелец пекарни",
    "Владелец цветочного магазина",
    "Владелец студии йоги"
]

# ===================== ВЫБОР МОДЕЛИ =====================

def select_model():
    """Позволяет выбрать модель для тестирования"""

    available_models = {
        # ========== КРУПНЫЕ МОДЕЛИ ==========
        "1": {
            "id": "qwen2.5-7b",
            "name": "Qwen/Qwen2.5-7B-Instruct",
            "description": "Qwen2.5-7B (Alibaba) - Лучшая в своем классе, отличный русский",
            "requires_agreement": True,
            "chat_template": "qwen"
        },
        "2": {
            "id": "gemma2-9b",
            "name": "google/gemma-2-9b-it",
            "description": "Google Gemma 2-9B (Google) - Современная модель от Google",
            "requires_agreement": True,
            "chat_template": "gemma"
        },
        "3": {
            "id": "mistral-7b",
            "name": "mistralai/Mistral-7B-Instruct-v0.1",
            "description": "Mistral 7B (Mistral AI) - Европейский лидер",
            "requires_agreement": False,
            "chat_template": "mistral"
        },
        "4": {
            "id": "phi-3-medium",
            "name": "microsoft/Phi-3-medium-4k-instruct",
            "description": "Microsoft Phi-3-medium (Microsoft) - 'Умная' компактная модель",
            "requires_agreement": True,
            "chat_template": "phi"
        },

        # ========== ЛОКАЛЬНЫЕ/СЛАБЫЕ МОДЕЛИ ==========
        "5": {
            "id": "phi-3-mini",
            "name": "microsoft/Phi-3-mini-4k-instruct",
            "description": "Microsoft Phi-3-mini (3.8B) - Король маленьких моделей",
            "requires_agreement": True,
            "chat_template": "phi"
        },
        "6": {
            "id": "gemma2-2b",
            "name": "google/gemma-2-2b-it",
            "description": "Google Gemma 2-2B (2.6B) - Компактная модель от Google",
            "requires_agreement": True,
            "chat_template": "gemma"
        },
        "7": {
            "id": "qwen2.5-1.5b",
            "name": "Qwen/Qwen2.5-1.5B-Instruct",
            "description": "Qwen2.5-1.5B (1.5B) - Промежуточный вариант",
            "requires_agreement": True,
            "chat_template": "qwen"
        },
        "8": {
            "id": "qwen2.5-0.5b",
            "name": "Qwen/Qwen2.5-0.5B-Instruct",
            "description": "Qwen2.5-0.5B (0.5B) - Для теста абсолютного минимума",
            "requires_agreement": True,
            "chat_template": "qwen"
        },
        "9": {
            "id": "phi-2",
            "name": "microsoft/phi-2",
            "description": "Microsoft Phi-2 (2.7B) - Предшественник Phi-3, все еще хороша",
            "requires_agreement": False,
            "chat_template": "phi"
        },
        "10": {
            "id": "olmo-1b",
            "name": "allenai/OLMo-1B-hf",
            "description": "Olmo 1B (1.2B) - Современная открытая модель",
            "requires_agreement": False,
            "chat_template": "default"
        },
        "11": {
            "id": "stablelm2-1.6b",
            "name": "stabilityai/stablelm-2-1_6b",
            "description": "StableLM 2 1.6B (1.6B) - Стабильная и быстрая",
            "requires_agreement": False,
            "chat_template": "default"
        },

        # ========== ДОПОЛНИТЕЛЬНЫЕ ОТКРЫТЫЕ ==========
        "12": {
            "id": "tinyllama",
            "name": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
            "description": "TinyLlama 1.1B - Очень популярная маленькая модель",
            "requires_agreement": False,
            "chat_template": "llama"
        },
        "13": {
            "id": "zephyr-7b",
            "name": "HuggingFaceH4/zephyr-7b-beta",
            "description": "Zephyr 7B - Fine-tuned Mistral",
            "requires_agreement": False,
            "chat_template": "mistral"
        }
    }

    print("=" * 60)
    print("🤖 ВЫБЕРИТЕ МОДЕЛЬ ДЛЯ ТЕСТИРОВАНИЯ")
    print("=" * 60)

    for key, model in available_models.items():
        print(f"{key}. {model['description']}")

    print("\nВведите номер модели (1-13):")
    choice = input("Ваш выбор: ").strip()

    if choice in available_models:
        selected = available_models[choice]
        print(f"\n✅ Выбрана модель: {selected['description']}")
        return selected
    else:
        print(f"\n⚠️ Неверный выбор. По умолчанию выбрана Qwen2.5-7B")
        return available_models["1"]

# ===================== ФУНКЦИИ ДЛЯ ТЕСТИРОВАНИЯ =====================

def test_single_model(model_info, use_pipeline=True):
    """Тестирование одной выбранной модели"""

    model_id = model_info["id"]
    model_name = model_info["name"]
    description = model_info["description"]

    print(f"\n🎯 Тестирую модель: {description}")
    print("-" * 50)

    results = {
        'metadata': {
            'product': PRODUCT_DESCRIPTION,
            'model': model_info,
            'test_cases': TEST_CASES,
            'timestamp': datetime.now().isoformat(),
            'mode': 'pipeline' if use_pipeline else 'direct'
        },
        'responses': {}
    }

    try:
        if use_pipeline:
            # Используем pipeline БЕЗ квантования
            print(f"🔄 Загружаю модель через pipeline...")

            pipe = pipeline(
                "text-generation",
                model=model_name,
                tokenizer=model_name,
                torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
                device_map="auto" if torch.cuda.is_available() else None,
                token=HF_TOKEN,
                # Убрал model_kwargs={"load_in_4bit": True}
            )

            print("✅ Модель загружена!")

        else:
            # Используем прямой подход БЕЗ квантования
            print(f"🔄 Загружаю модель напрямую...")

            tokenizer = AutoTokenizer.from_pretrained(
                model_name,
                token=HF_TOKEN,
                trust_remote_code=True
            )

            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                torch_dtype=torch.float16,
                device_map="auto",
                token=HF_TOKEN,
                # Убрал load_in_4bit=True
                trust_remote_code=True
            )

            print("✅ Модель загружена!")

    except Exception as e:
        print(f"❌ Ошибка загрузки модели: {str(e)[:200]}")
        return None

    # Тестируем для каждого кейса
    for i, test_case in enumerate(TEST_CASES, 1):
        print(f"\n[{i}/{len(TEST_CASES)}] Тест-кейс: {test_case}")

        prompt = PROMPT_TEMPLATE.replace("{target_audience}", test_case)

        try:
            if use_pipeline:
                # Генерация через pipeline
                response = pipe(
                    prompt,
                    max_new_tokens=350,  # Уменьшил для скорости
                    temperature=0.7,
                    do_sample=True,
                    top_p=0.9
                )[0]['generated_text']

            else:
                # Генерация напрямую
                # Форматируем промпт
                if "qwen" in model_id:
                    messages = [
                        {"role": "system", "content": "Ты - опытный AI-ассистент по продажам."},
                        {"role": "user", "content": prompt}
                    ]
                    text = tokenizer.apply_chat_template(
                        messages,
                        tokenize=False,
                        add_generation_prompt=True
                    )
                elif "gemma" in model_id:
                    text = f"<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n"
                elif "mistral" in model_id:
                    text = f"[INST] {prompt} [/INST]"
                else:
                    text = prompt

                # Генерация
                inputs = tokenizer(text, return_tensors="pt").to(model.device)

                with torch.no_grad():
                    outputs = model.generate(
                        **inputs,
                        max_new_tokens=150,  # Уменьшил
                        temperature=0.7,
                        do_sample=True,
                        top_p=0.9
                    )

                # Декодирование
                response = tokenizer.decode(outputs[0], skip_special_tokens=True)
                response = response.replace(text, "").strip()

            # Очистка ответа
            response = clean_response(response, prompt, model_id)
            results['responses'][test_case] = response

            print(f"   📝 Ответ: {response[:100]}...")

        except Exception as e:
            error_msg = f"❌ Ошибка генерации: {str(e)[:100]}"
            results['responses'][test_case] = error_msg
            print(f"   {error_msg}")

    return results

def clean_response(text: str, original_prompt: str, model_id: str) -> str:
    """Очистка ответа от лишних технических частей"""
    if not isinstance(text, str):
        return str(text)

    # Удаляем дублирование промпта
    if original_prompt in text:
        text = text.replace(original_prompt, "").strip()

    # Удаляем технические теги
    if "qwen" in model_id:
        text = text.replace("<|im_start|>", "").replace("<|im_end|>", "").replace("assistant", "")
    elif "gemma" in model_id:
        text = text.replace("<start_of_turn>", "").replace("<end_of_turn>", "").replace("model", "")
    elif "mistral" in model_id:
        text = text.replace("[INST]", "").replace("[/INST]", "")

    # Удаляем лишние пробелы
    text = ' '.join(text.split())

    # Обрезаем слишком длинные ответы
    if len(text) > 500:
        text = text[:500] + "..."

    return text.strip()

# ===================== АНАЛИЗ И ВЫВОД РЕЗУЛЬТАТОВ =====================

def display_results(results):
    """Красивый вывод результатов"""
    if not results:
        print("\n❌ Нет результатов для отображения")
        return

    model_info = results['metadata']['model']

    print("\n" + "=" * 70)
    print(f"📊 РЕЗУЛЬТАТЫ ТЕСТИРОВАНИЯ: {model_info['description']}")
    print("=" * 70)

    for test_case, response in results['responses'].items():
        print(f"\n🎯 {test_case}:")
        print("-" * 40)
        print(response)
        print()

    # Сохраняем в файл
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    model_id = model_info['id']
    filename = f"{model_id}_results_{timestamp}.json"

    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    print(f"\n✅ Результаты сохранены в файл: {filename}")

    # Создаем HTML отчет
    create_html_report(results, filename.replace('.json', '.html'))

def create_html_report(results, filename):
    """Создание HTML отчета"""
    model_info = results['metadata']['model']

    html_content = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="UTF-8">
        <title>Результаты тестирования {model_info['description']}</title>
        <style>
            body {{ font-family: Arial, sans-serif; padding: 20px; max-width: 800px; margin: 0 auto; }}
            h1, h2 {{ color: #333; }}
            .test-case {{
                border: 1px solid #ddd;
                padding: 20px;
                margin: 15px 0;
                border-radius: 5px;
                background: #f9f9f9;
            }}
            .response {{
                white-space: pre-wrap;
                background: white;
                padding: 15px;
                border-radius: 3px;
                border-left: 4px solid #4CAF50;
            }}
            .info {{
                background: #e3f2fd;
                padding: 10px;
                border-radius: 5px;
                margin: 10px 0;
            }}
        </style>
    </head>
    <body>
        <h1>🤖 Результаты тестирования AI-модели</h1>

        <div class="info">
            <p><strong>Модель:</strong> {model_info['description']}</p>
            <p><strong>Дата тестирования:</strong> {datetime.now().strftime('%d.%m.%Y %H:%M')}</p>
            <p><strong>Продукт:</strong> {PRODUCT_DESCRIPTION[:100]}...</p>
        </div>
    """

    for test_case, response in results['responses'].items():
        html_content += f"""
        <div class="test-case">
            <h2>🎯 {test_case}</h2>
            <div class="response">{response}</div>
        </div>
        """

    html_content += """
    </body>
    </html>
    """

    with open(filename, 'w', encoding='utf-8') as f:
        f.write(html_content)

    print(f"📁 HTML отчет сохранен: {filename}")

def analyze_single_model(results):
    """Анализ результатов одной модели"""
    print("\n" + "=" * 70)
    print("📈 АНАЛИЗ КАЧЕСТВА СООБЩЕНИЙ")
    print("=" * 70)

    total_score = 0
    total_tests = len(results['responses'])

    for test_case, response in results['responses'].items():
        if not isinstance(response, str):
            continue

        words = response.split()
        word_count = len(words)

        # Проверяем критерии
        checks = {
            'greeting': any(word in response.lower() for word in
                           ['привет', 'здравствуйте', 'добрый', 'здравствуй']),
            'personalization': any(word in response.lower() for word in
                                  [test_case.lower().replace('владелец ', ''), 'ваш', 'вас', 'вам']),
            'problem': any(word in response.lower() for word in
                          ['проблем', 'трудност', 'сложност', 'нет клиент', 'упускаешь']),
            'solution': any(word in response.lower() for word in
                           ['сайт', 'сайта', 'визитка', 'интернет', 'онлайн']),
            'cta': any(word in response.lower() for word in
                      ['звонок', 'свяжись', 'напиши', 'ответь', 'обсудим']),
            'natural_length': 30 <= word_count <= 120
        }

        score = sum(checks.values())
        total_score += score

        print(f"\n📋 {test_case}:")
        print(f"   Слов: {word_count}, Баллов: {score}/6")

        for check_name, passed in checks.items():
            icon = "✅" if passed else "❌"
            names = {
                'greeting': 'Приветствие',
                'personalization': 'Персонализация',
                'problem': 'Проблема клиента',
                'solution': 'Решение (сайт)',
                'cta': 'Призыв к действию',
                'natural_length': 'Оптимальная длина'
            }
            print(f"   {icon} {names.get(check_name, check_name)}")

    avg_score = total_score / total_tests if total_tests > 0 else 0
    print(f"\n📊 Средний балл: {avg_score:.1f}/6")

    if avg_score >= 4:
        print("🎉 Отличные результаты! Сообщения соответствуют критериям.")
    elif avg_score >= 3:
        print("⚠️ Средние результаты. Есть над чем поработать.")
    else:
        print("❌ Низкие результаты. Нужно улучшать промпт или выбрать другую модель.")

# ===================== ЗАПУСК ТЕСТИРОВАНИЯ =====================
if __name__ == "__main__":
    print("\n" + "=" * 70)
    print("🚀 СИСТЕМА ТЕСТИРОВАНИЯ AI-МОДЕЛЕЙ ДЛЯ ХОЛОДНЫХ РАССЫЛОК")
    print("=" * 70)

    # Выбираем модель
    model_info = select_model()

    # Запускаем тестирование
    print("\n" + "=" * 70)
    print("⏳ Начинаю тестирование... Это может занять несколько минут.")
    print("=" * 70)

    results = test_single_model(model_info, use_pipeline=True)

    if results:
        # Показываем результаты
        display_results(results)

        # Анализируем результаты
        analyze_single_model(results)

        print("\n" + "=" * 70)
        print("✅ ТЕСТИРОВАНИЕ ЗАВЕРШЕНО!")
        print("=" * 70)

        # Предложение протестировать другую модель
        print("\n🔄 Хотите протестировать другую модель?")
        print("Просто запустите ячейку снова и выберите другую модель.")
    else:
        print("\n❌ Тестирование не удалось. Проверьте:")
        print("   1. Доступ к интернету")
        print("   2. Достаточно ли памяти в Colab")
        print("   3. Корректность токена Hugging Face")

✅ Успешная авторизация в Hugging Face!

🚀 СИСТЕМА ТЕСТИРОВАНИЯ AI-МОДЕЛЕЙ ДЛЯ ХОЛОДНЫХ РАССЫЛОК
🤖 ВЫБЕРИТЕ МОДЕЛЬ ДЛЯ ТЕСТИРОВАНИЯ
1+. Qwen2.5-7B (Alibaba) - Лучшая в своем классе, отличный русский
2-. Google Gemma 2-9B (Google) - Современная модель от Google
3+. Mistral 7B (Mistral AI) - Европейский лидер
4-(тяжелая). Microsoft Phi-3-medium (Microsoft) - 'Умная' компактная модель
5+. Microsoft Phi-3-mini (3.8B) - Король маленьких моделей
6-. Google Gemma 2-2B (2.6B) - Компактная модель от Google
7+. Qwen2.5-1.5B (1.5B) - Промежуточный вариант
8+. Qwen2.5-0.5B (0.5B) - Для теста абсолютного минимума
9+. Microsoft Phi-2 (2.7B) - Предшественник Phi-3, все еще хороша
10+. Olmo 1B (1.2B) - Современная открытая модель
11+. StableLM 2 1.6B (1.6B) - Стабильная и быстрая
12+. TinyLlama 1.1B - Очень популярная маленькая модель
13+. Zephyr 7B - Fine-tuned Mistral

Введите номер модели (1-13):
Ваш выбор: 6-

✅ Выбрана модель: Google Gemma 2-2B (2.6B) - Компактная модель от Google

⏳ Начинаю тести